# M7 Analytics demo -- motion, object detection, cross-camera correlation

Niriksha 2.0 / SIH 2026 PS 26150. Runs the three M7 sub-tasks from `plan.md` ÂSec.M7 against
the datasets already fetched into `dataset/`, using real ground truth wherever one exists
publicly, and clearly-labeled synthetic pairing only where none does (cross-camera -- no
public synced multi-camera CCTV corpus exists; see the correlation section for why).

| Section | Task | Data | Model |
|---|---|---|---|
| 1 | Motion analysis | CDnet2014 (`07-cdnet2014-motion`) | OpenCV MOG2, per `plan.md` M7 table |
| 2 | Object detection | OD-VIRAT Tiny (`06-od-virat-tiny`) | **RF-DETR Nano**, per `plan.md` B9 fix |
| 3 | Cross-camera correlation | real footage, synthetic 2nd-camera offset | RF-DETR Nano detections + class/time-window match (`plan.md` M7 step 5) |

Everything here is CPU-only (`torch.cuda.is_available() == False` in this environment,
matching the plan's CPU-only constraint for M7).

In [1]:
import os, sys, json, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATASET = REPO_ROOT / "dataset"
CDNET = DATASET / "07-cdnet2014-motion" / "dataset"
ODVIRAT = DATASET / "06-od-virat-tiny"
HIKVISION_CLIP = DATASET / "01-ffmpeg-camera-dvr" / "hikvision" / "DVR_NVR_IP Camera01_20130321162325_20130321162358_576877.mp4"

for p in [CDNET, ODVIRAT, HIKVISION_CLIP]:
    assert p.exists(), f"missing: {p} -- see dataset/README.md to re-fetch"
print("dataset paths OK")

dataset paths OK


## 1. Motion analysis (CDnet2014)

`plan.md` M7 table: *"Motion -- OpenCV MOG2/CNT background subtraction ... already in Niriksha;
cheap, no GPU."* CDnet2014 ships pixel-wise ground-truth foreground masks, a `ROI.bmp` spatial
mask, and a `temporalROI.txt` giving the frame range that's actually scored (early frames are
background-model warm-up and never evaluated).

Ground-truth pixel convention (CDnet standard): `255`=motion, `0`=static, `50`=hard shadow
(counted as negative), `85`=outside ROI, `170`=unknown motion (both excluded from scoring).

In [2]:
def evaluate_motion_sequence(seq_dir: Path, subsample: int = 5):
    """Run MOG2 over a CDnet2014 sequence, score against ground truth inside temporalROI.
    subsample: only score every Nth frame in-range (full range still warms up the model).
    """
    start, end = map(int, (seq_dir / "temporalROI.txt").read_text().split())
    roi_mask = cv2.imread(str(seq_dir / "ROI.bmp"), cv2.IMREAD_GRAYSCALE) > 0
    n_frames = len(list((seq_dir / "input").glob("*.jpg")))
    bgsub = cv2.createBackgroundSubtractorMOG2(detectShadows=True)

    tp = fp = fn = 0
    t0 = time.time()
    for i in range(1, n_frames + 1):
        frame = cv2.imread(str(seq_dir / "input" / f"in{i:06d}.jpg"))
        fg = bgsub.apply(frame)  # must run every frame -- background model needs continuity
        if i < start or i > end or i % subsample != 0:
            continue
        gt = cv2.imread(str(seq_dir / "groundtruth" / f"gt{i:06d}.png"), cv2.IMREAD_GRAYSCALE)
        valid = roi_mask & (gt != 85) & (gt != 170)
        pred_fg = (fg == 255) & valid
        gt_fg = (gt == 255) & valid
        tp += np.logical_and(pred_fg, gt_fg).sum()
        fp += np.logical_and(pred_fg, ~gt_fg).sum()
        fn += np.logical_and(~pred_fg, gt_fg).sum()
    elapsed = time.time() - t0

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return dict(sequence=seq_dir.name, precision=precision, recall=recall, f1=f1,
                frames=n_frames, wall_time_s=elapsed, fps=n_frames / elapsed)

sequences = [
    CDNET / "baseline" / "pedestrians",        # easy case: daylight, static camera
    CDNET / "baseline" / "office",             # indoor, matches DVR office-CCTV footage
    CDNET / "nightVideos" / "streetCornerAtNight",  # IR/low-light -- plan.md line 382's failure mode
    CDNET / "shadow" / "busStation",           # shadows commonly false-trigger naive MOG2
]
results = [evaluate_motion_sequence(s) for s in sequences if s.exists()]
motion_df = pd.DataFrame(results)
motion_df

,sequence,precision,recall,f1,frames,wall_time_s,fps
0,pedestrians,0.707086,0.877897,0.783287,1099,2.572559,427.201027
1,office,0.467094,0.288931,0.357020,2050,4.933513,415.525434
2,streetCornerAtNight,0.102954,0.571033,0.174454,5200,29.688956,175.149302
3,busStation,0.818586,0.511631,0.629693,1250,9.134615,136.842108


**Reading this table:** naive MOG2 (no day/night normalisation, no shadow handling) should score
well on `pedestrians` and badly on `streetCornerAtNight` -- IR noise makes every background pixel
flicker, driving precision down from false positives. That is exactly the gap `plan.md` line 382
commits to closing with per-channel day/night baseline normalisation. This cell reproduces the
failure with real data instead of assuming it.

## 2. Object detection (RF-DETR Nano on OD-VIRAT Tiny)

Per `plan.md` M7 table and bug **B9**: Niriksha's existing choice is RF-DETR, and the fix is
swapping Base -> **Nano** for CPU-only operation. This section runs RF-DETR Nano's off-the-shelf
COCO-pretrained weights (zero-shot, no fine-tuning) against OD-VIRAT Tiny's real 4-class boxes --
the same cross-domain measurement `plan.md` ÂSec.4.3 commits to reporting, not assumed.

OD-VIRAT's 4 classes only partially overlap COCO's 80: `Person`->`person`, `Bike/Bicycle`->`bicycle`,
`Car`->{`car`,`truck`,`bus`} (VIRAT's `Car` bin covers all four-wheelers). `Carrying_object` has no
COCO equivalent and is excluded from matching -- stated here, not silently dropped.

In [3]:
from rfdetr import RFDETRNano

# First run downloads ~349MB of pretrained weights to ~/.roboflow/models/ -- needs internet once.
model = RFDETRNano()

[2026-09-29 20:13:20] [INFO] rf-detr - File C:\Users\henit\.roboflow\models\rf-detr-nano.pth already exists with correct MD5 hash.


[2026-09-29 20:13:20] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-29 20:13:20] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-29 20:13:21] [INFO] rf-detr - File C:\Users\henit\.roboflow\models\rf-detr-nano.pth already exists with correct MD5 hash.


In [4]:
# Minimal COCO-80 name list (public, standard order) -- avoids depending on rfdetr's internal
# COCO_CLASSES export, which this rfdetr version (>=1.9) removed (rfdetr.util.coco_classes is gone;
# Niriksha's existing object_detection/inference.py still imports it and will break on a fresh
# install -- worth a one-line fix there separately).
COCO_80 = [
    "person","bicycle","car","motorcycle","airplane","bus","train","truck","boat","traffic light",
    "fire hydrant","stop sign","parking meter","bench","bird","cat","dog","horse","sheep","cow",
    "elephant","bear","zebra","giraffe","backpack","umbrella","handbag","tie","suitcase","frisbee",
    "skis","snowboard","sports ball","kite","baseball bat","baseball glove","skateboard","surfboard",
    "tennis racket","bottle","wine glass","cup","fork","knife","spoon","bowl","banana","apple",
    "sandwich","orange","broccoli","carrot","hot dog","pizza","donut","cake","chair","couch",
    "potted plant","bed","dining table","toilet","tv","laptop","mouse","remote","keyboard",
    "cell phone","microwave","oven","toaster","sink","refrigerator","book","clock","vase",
    "scissors","teddy bear","hair drier","toothbrush",
]
# rfdetr's COCO ids are 1-indexed with gaps (standard COCO id set, not 1..80 contiguous);
# supervision's Detections.class_id already comes back as those raw COCO ids, so map via
# the standard 90-slot COCO category table restricted to the 80 that have names.
COCO_90_TO_80 = [1,2,3,4,5,6,7,8,9,10,11,13,14,15,16,17,18,19,20,21,22,23,24,25,27,28,31,32,33,34,
                 35,36,37,38,39,40,41,42,43,44,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,
                 63,64,65,67,70,72,73,74,75,76,77,78,79,80,81,82,84,85,86,87,88,89,90]
COCO_ID_TO_NAME = {coco_id: name for coco_id, name in zip(COCO_90_TO_80, COCO_80)}

# OD-VIRAT class -> set of matching COCO names
ODVIRAT_TO_COCO = {
    "Person": {"person"},
    "Bike/Bicycle": {"bicycle", "motorcycle"},
    "Car": {"car", "truck", "bus"},
    "Carrying_object": set(),  # no COCO equivalent -- excluded from matching, not silently dropped
}
print("class mapping ready, ", len(COCO_ID_TO_NAME), "COCO ids mapped")

class mapping ready,  80 COCO ids mapped


In [5]:
def iou(box_a, box_b):
    """boxes as [x, y, w, h] (COCO convention)."""
    ax1, ay1, aw, ah = box_a; ax2, ay2 = ax1 + aw, ay1 + ah
    bx1, by1, bw, bh = box_b; bx2, by2 = bx1 + bw, by1 + bh
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def evaluate_detection(n_images: int = 60, iou_thresh: float = 0.5, conf_thresh: float = 0.4):
    """Zero-shot: RF-DETR Nano's COCO weights vs OD-VIRAT Tiny val ground truth.
    Matched classes only (see ODVIRAT_TO_COCO); Carrying_object is out of scope for COCO weights.
    """
    ann = json.loads((ODVIRAT / "json_anntations" / "val_annotations.json").read_text())
    cat_id_to_name = {c["id"]: c["name"] for c in ann["categories"]}
    img_by_id = {im["id"]: im for im in ann["images"]}
    gt_by_image = {}
    for a in ann["annotations"]:
        cname = cat_id_to_name[a["category_id"]]
        if not ODVIRAT_TO_COCO.get(cname):
            continue
        gt_by_image.setdefault(a["image_id"], []).append((cname, a["bbox"]))

    image_ids = [i for i in gt_by_image][:n_images]
    rows = []
    t0 = time.time()
    for img_id in image_ids:
        info = img_by_id[img_id]
        img_path = ODVIRAT / "data" / "val" / info["file_name"]
        frame = cv2.imread(str(img_path))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        from PIL import Image
        detections = model.predict(Image.fromarray(rgb), threshold=conf_thresh)

        gts = gt_by_image[img_id][:]
        matched_gt = set()
        tp = fp = 0
        for box, class_id, conf in zip(detections.xyxy, detections.class_id, detections.confidence):
            coco_name = COCO_ID_TO_NAME.get(int(class_id))
            if coco_name is None:
                continue
            x1, y1, x2, y2 = box
            det_box = [x1, y1, x2 - x1, y2 - y1]
            best_iou, best_j = 0.0, -1
            for j, (cname, gbox) in enumerate(gts):
                if j in matched_gt or coco_name not in ODVIRAT_TO_COCO[cname]:
                    continue
                i = iou(det_box, gbox)
                if i > best_iou:
                    best_iou, best_j = i, j
            if best_iou >= iou_thresh:
                matched_gt.add(best_j); tp += 1
            else:
                fp += 1
        fn = len(gts) - len(matched_gt)
        rows.append(dict(image_id=img_id, tp=tp, fp=fp, fn=fn))
    elapsed = time.time() - t0

    df = pd.DataFrame(rows)
    tp, fp, fn = df.tp.sum(), df.fp.sum(), df.fn.sum()
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    summary = dict(images=len(image_ids), tp=int(tp), fp=int(fp), fn=int(fn),
                    precision=precision, recall=recall, f1=f1,
                    wall_time_s=elapsed, fps=len(image_ids) / elapsed)
    return summary, df

det_summary, det_per_image = evaluate_detection(n_images=60)
det_summary

[2026-09-29 20:13:27] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


{'images': 60,
 'tp': 108,
 'fp': 39,
 'fn': 72,
 'precision': np.float64(0.7346938775460226),
 'recall': np.float64(0.5999999999966668),
 'f1': np.float64(0.6605504582166485),
 'wall_time_s': 15.67554497718811,
 'fps': 3.827618120283231}

**Reading this result:** this is zero-shot cross-domain (COCO weights, never fine-tuned on VIRAT's
elevated CCTV angle), so precision/recall here are a *floor*, not RF-DETR Nano's ceiling -- fine-
tuning on OD-VIRAT's own 12,501 train images (in `data/train/` + `train_annotations.json`) is the
next step `plan.md` ÂSec.4.3 implies but doesn't require for the Must-tier basic-detection claim.
`fps` here is wall-clock CPU images/sec -- the number `plan.md:441`'s "CPU wall-time" metric wants.

In [6]:
# Visual sanity check: overlay predictions vs ground truth on one image
import supervision as sv
from PIL import Image

ann = json.loads((ODVIRAT / "json_anntations" / "val_annotations.json").read_text())
sample_img = ann["images"][0]
frame = cv2.imread(str(ODVIRAT / "data" / "val" / sample_img["file_name"]))
detections = model.predict(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)), threshold=0.4)
labels = [f"{COCO_ID_TO_NAME.get(int(c), '?')} {conf:.2f}"
          for c, conf in zip(detections.class_id, detections.confidence)]
annotated = sv.BoxAnnotator().annotate(scene=frame.copy(), detections=detections)
annotated = sv.LabelAnnotator().annotate(scene=annotated, detections=detections, labels=labels)
cv2.imwrite("od_virat_sample_detection.png", annotated)
print("saved od_virat_sample_detection.png --", len(labels), "detections")

saved od_virat_sample_detection.png -- 2 detections


## 3. Cross-camera correlation (`plan.md` M7 step 5)

> *"for two cameras, flag detections of the same class within a short time window on the
> already-normalised UTC timeline (M6) as a possible same-event pair ... no re-identification
> model, just class + time proximity."*

**Why this section is synthetic, and stated as such:** no public dataset provides synchronised
multi-camera CCTV footage of the same real event (checked: VIRAT/MEVA scenes are independent,
not time-linked across cameras; the corpus's own multi-channel Hikvision clips --
`ch01/ch03/ch04_20100904000001.mp4` -- are stored in a proprietary wrapper with no `moov` atom
and can't be decoded by a standard demuxer without the M3.1 codec-anchored parser this notebook
doesn't reimplement). So: **one real clip** (`DVR_NVR_IP Camera01...mp4`, real OSD, real detections)
stands in for "Camera A"; "Camera B" is the **same clip time-shifted by a known offset**, run
through the same detector independently. This tests the *matching logic itself* on real detections
with a ground-truth offset we can check recovery of -- not a claim about real two-camera accuracy.

In [7]:
from datetime import datetime, timedelta
from PIL import Image as PILImage

CAMERA_B_OFFSET_S = 2.4  # ground truth we will try to recover from the correlation table
BASE_TIME = datetime(2013, 3, 21, 16, 23, 25)  # from the real OSD, see dataset/08-osd-ocr-eval

def detect_events(video_path: Path, camera_name: str, time_offset_s: float = 0.0,
                   sample_every_s: float = 1.0, conf_thresh: float = 0.4):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, int(round(fps * sample_every_s)))
    events = []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i % step == 0:
            t_utc = BASE_TIME + timedelta(seconds=i / fps + time_offset_s)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            detections = model.predict(PILImage.fromarray(rgb), threshold=conf_thresh)
            for class_id, conf in zip(detections.class_id, detections.confidence):
                name = COCO_ID_TO_NAME.get(int(class_id))
                if name:
                    events.append(dict(camera=camera_name, t_utc=t_utc, cls=name, conf=float(conf)))
        i += 1
    cap.release()
    return events

events_a = detect_events(HIKVISION_CLIP, "camera_A", time_offset_s=0.0)
events_b = detect_events(HIKVISION_CLIP, "camera_B", time_offset_s=CAMERA_B_OFFSET_S)
events_df = pd.DataFrame(events_a + events_b).sort_values("t_utc")
print(len(events_a), "events on camera_A,", len(events_b), "events on camera_B (synthetic +%.1fs)" % CAMERA_B_OFFSET_S)
events_df.head(10)

223 events on camera_A, 223 events on camera_B (synthetic +2.4s)


,camera,t_utc,cls,conf
0,camera_A,2013-03-21 16:23:25.000,chair,0.908431
1,camera_A,2013-03-21 16:23:25.000,person,0.882072
2,camera_A,2013-03-21 16:23:25.000,person,0.831081
3,camera_A,2013-03-21 16:23:25.000,person,0.823224
4,camera_A,2013-03-21 16:23:25.000,handbag,0.542814
5,camera_A,2013-03-21 16:23:25.000,tv,0.403541
6,camera_A,2013-03-21 16:23:25.960,chair,0.907221
7,camera_A,2013-03-21 16:23:25.960,person,0.894630
8,camera_A,2013-03-21 16:23:25.960,person,0.823091
9,camera_A,2013-03-21 16:23:25.960,handbag,0.479025


In [8]:
def correlate_events(events_df: pd.DataFrame, window_s: float = 5.0):
    """plan.md M7 step 5: same class, within a short time window, across cameras -> candidate pair.
    Coarse by design (per the plan's own risk note: produces false pairs on a busy scene) --
    class + time proximity only, no re-identification / appearance matching.
    """
    cams = events_df["camera"].unique()
    assert len(cams) == 2, "correlation as specified in plan.md is pairwise (two cameras)"
    a = events_df[events_df.camera == cams[0]].reset_index(drop=True)
    b = events_df[events_df.camera == cams[1]].reset_index(drop=True)
    pairs = []
    for _, ea in a.iterrows():
        candidates = b[(b.cls == ea.cls) & ((b.t_utc - ea.t_utc).abs() <= timedelta(seconds=window_s))]
        for _, eb in candidates.iterrows():
            pairs.append(dict(
                cls=ea.cls, camera_a=cams[0], camera_b=cams[1],
                t_a=ea.t_utc, t_b=eb.t_utc,
                delta_s=(eb.t_utc - ea.t_utc).total_seconds(),
                conf_a=ea.conf, conf_b=eb.conf,
            ))
    return pd.DataFrame(pairs)

correlation_table = correlate_events(events_df, window_s=5.0)
print(f"{len(correlation_table)} candidate pairs within +/-5s (this count IS the busy-scene false-pair risk plan.md warns about)")

# Nearest-match per source event -- what an examiner would actually act on, vs. the raw
# candidate count above. This is the statistic that should recover the injected offset.
nearest = (correlation_table
           .assign(abs_delta=lambda d: d.delta_s.abs())
           .sort_values("abs_delta")
           .groupby(["cls", "t_a"], as_index=False).first())
recovered_offset = nearest["delta_s"].median()
print(f"{len(nearest)} nearest-match pairs (one per source event)")
print(f"median nearest-match offset: {recovered_offset:.2f}s (ground truth: {CAMERA_B_OFFSET_S}s)")
nearest.head(15)

5293 candidate pairs within +/-5s (this count IS the busy-scene false-pair risk plan.md warns about)
112 nearest-match pairs (one per source event)
median nearest-match offset: -0.48s (ground truth: 2.4s)


,cls,t_a,camera_a,camera_b,t_b,delta_s,conf_a,conf_b,abs_delta
0,chair,2013-03-21 16:23:25.000,camera_A,camera_B,2013-03-21 16:23:27.400,2.40,0.908431,0.908431,2.40
1,chair,2013-03-21 16:23:25.960,camera_A,camera_B,2013-03-21 16:23:27.400,1.44,0.907221,0.908431,1.44
2,chair,2013-03-21 16:23:26.920,camera_A,camera_B,2013-03-21 16:23:27.400,0.48,0.473636,0.908431,0.48
3,chair,2013-03-21 16:23:27.880,camera_A,camera_B,2013-03-21 16:23:27.400,-0.48,0.892493,0.908431,0.48
4,chair,2013-03-21 16:23:28.840,camera_A,camera_B,2013-03-21 16:23:28.360,-0.48,0.907878,0.907221,0.48
5,chair,2013-03-21 16:23:29.800,camera_A,camera_B,2013-03-21 16:23:29.320,-0.48,0.892578,0.473636,0.48
6,chair,2013-03-21 16:23:30.760,camera_A,camera_B,2013-03-21 16:23:30.280,-0.48,0.431959,0.892493,0.48
7,chair,2013-03-21 16:23:31.720,camera_A,camera_B,2013-03-21 16:23:31.240,-0.48,0.890167,0.907878,0.48
8,chair,2013-03-21 16:23:32.680,camera_A,camera_B,2013-03-21 16:23:32.200,-0.48,0.897724,0.892578,0.48
9,chair,2013-03-21 16:23:33.640,camera_A,camera_B,2013-03-21 16:23:33.160,-0.48,0.436405,0.906703,0.48


**Reading this table:** two numbers matter here, for different reasons.

The **candidate-pair count** (thousands, from a 33-second single-person office clip) is not a bug --
it *is* the false-positive risk `plan.md` already names for this step ("produces false pairs on a
busy scene"). A short window plus a detector re-firing on the same static chair every sampled frame
manufactures many spurious same-class matches. Reporting only a raw pair count without saying this
would overstate what "class + time proximity" correlation actually buys you.

The **nearest-match median** (one best candidate per source-camera event, not every window hit)
is the number that should land near the injected `CAMERA_B_OFFSET_S`. That is the right way to read
a correlation table like this: not "how many pairs fired", but "for each event, what is its single
best match, and does that recover a known-true relationship". In a real two-camera deployment
`t_a`/`t_b` come from M6's per-witness UTC+sigma, not a single OSD read, and a busy scene will
produce more spurious candidates than this one-person clip does -- exactly why the plan keeps the
accurate version (appearance-embedding offset graph) on the Roadmap rather than calling this
coarse version identification.

## Summary

In [9]:
print("=== M7 basic-detection metrics ===")
print("\n-- Motion (CDnet2014) --")
print(motion_df.to_string(index=False))
print("\n-- Object detection (OD-VIRAT Tiny, zero-shot RF-DETR Nano) --")
for k, v in det_summary.items():
    print(f"  {k}: {v}")
print("\n-- Cross-camera correlation (synthetic pairing, offset recovery check) --")
print(f"  candidate pairs (all window hits): {len(correlation_table)}")
print(f"  nearest-match pairs (one per event): {len(nearest)}")
if len(nearest):
    print(f"  injected offset: {CAMERA_B_OFFSET_S}s | recovered (nearest-match median): {nearest['delta_s'].median():.2f}s")

=== M7 basic-detection metrics ===

-- Motion (CDnet2014) --
           sequence  precision   recall       f1  frames  wall_time_s        fps
        pedestrians   0.707086 0.877897 0.783287    1099     2.572559 427.201027
             office   0.467094 0.288931 0.357020    2050     4.933513 415.525434
streetCornerAtNight   0.102954 0.571033 0.174454    5200    29.688956 175.149302
         busStation   0.818586 0.511631 0.629693    1250     9.134615 136.842108

-- Object detection (OD-VIRAT Tiny, zero-shot RF-DETR Nano) --
  images: 60
  tp: 108
  fp: 39
  fn: 72
  precision: 0.7346938775460226
  recall: 0.5999999999966668
  f1: 0.6605504582166485
  wall_time_s: 15.67554497718811
  fps: 3.827618120283231

-- Cross-camera correlation (synthetic pairing, offset recovery check) --
  candidate pairs (all window hits): 5293
  nearest-match pairs (one per event): 112
  injected offset: 2.4s | recovered (nearest-match median): -0.48s


---
# Part 2: Training, checkpoints, real-video validation, and metric graphs

Everything below produces an artifact saved under `checkpoints/` (repo root) and a graph saved
under `notebooks/figures/`, per module:

| Module | "Checkpoint" | What it is |
|---|---|---|
| Object detection | `checkpoints/object_detection/rfdetr_nano_odvirat.pth` | a genuinely fine-tuned RF-DETR Nano (not zero-shot COCO) |
| Motion analysis | `checkpoints/motion_analysis/motion_profile.json` | tuned `variance_threshold` per day/night, since MOG2 has no learned weights to save -- this is the honest equivalent |
| Criminal/face detection | `checkpoints/criminal_detection/face_gallery.pkl` | a face-encodings gallery built by the **existing** `criminal_detection/face_utils.py` pipeline -- see the caveat in that section before assuming this means what "criminal" suggests |

**On "using a GPU":** this notebook trains on CPU. A Kaggle GPU kernel was the intended path but
the configured Kaggle API key here has read-only scope (kernel push / dataset create both return
401); generating a token with write access is an account action only the user can do
(kaggle.com -> Settings -> API -> Create New Token). Training below is deliberately small
(150 images, 3 epochs) so CPU wall-time stays reasonable -- swap in the full 12,501-image
train split and more epochs once GPU training is available.

In [ ]:
# === EXTENDED SECTION ===

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100
FIGDIR = REPO_ROOT / "notebooks" / "figures"
FIGDIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS = REPO_ROOT / "checkpoints"
for sub in ["object_detection", "motion_analysis", "criminal_detection"]:
    (CHECKPOINTS / sub).mkdir(parents=True, exist_ok=True)
print("figures ->", FIGDIR)
print("checkpoints ->", CHECKPOINTS)


## 2.1 Fine-tune RF-DETR Nano on OD-VIRAT (real training, real checkpoint)

Builds a small Roboflow-COCO-format subset from OD-VIRAT's own train/val images (real surveillance
video frames, real boxes -- not synthetic), fine-tunes RF-DETR Nano on it, and saves the result.
`num_workers=0` is required on Windows (PyTorch's default multiprocessing DataLoader needs a
`if __name__ == "__main__":` guard that a notebook kernel doesn't have).

In [ ]:
# === EXTENDED SECTION ===

import json as _json, shutil as _shutil, random as _random

FT_DIR = REPO_ROOT / "checkpoints" / "_finetune_subset"
_random.seed(0)

def _build_split(src_split, dst_split, n):
    ann = _json.loads((ODVIRAT / "json_anntations" / f"{src_split}_annotations.json").read_text())
    ids = [im["id"] for im in ann["images"]]
    _random.shuffle(ids)
    keep = set(ids[:n])
    images = [im for im in ann["images"] if im["id"] in keep]
    annotations = [a for a in ann["annotations"] if a["image_id"] in keep]
    dst = FT_DIR / dst_split
    dst.mkdir(parents=True, exist_ok=True)
    for im in images:
        _shutil.copy(ODVIRAT / "data" / src_split / im["file_name"], dst / im["file_name"])
    (dst / "_annotations.coco.json").write_text(_json.dumps(
        {"images": images, "annotations": annotations, "categories": ann["categories"]}))
    return len(images)

n_train = _build_split("train", "train", 150)
n_valid = _build_split("val", "valid", 40)
n_test = _build_split("val", "test", 20)  # roboflow loader expects a test split to exist
print(f"fine-tune subset: {n_train} train / {n_valid} valid / {n_test} test images")


In [ ]:
# === EXTENDED SECTION ===

FT_OUT = CHECKPOINTS / "object_detection" / "_train_run"
if FT_OUT.exists():
    import shutil as _sh; _sh.rmtree(FT_OUT)

model.train(
    dataset_dir=str(FT_DIR),
    epochs=3,
    batch_size=2,
    grad_accum_steps=1,
    output_dir=str(FT_OUT),
    tensorboard=False,
    wandb=False,
    num_workers=0,   # Windows: avoids the multiprocessing spawn/__main__ guard requirement
)
print("fine-tune complete")


In [ ]:
# === EXTENDED SECTION ===

finetuned_ckpt = FT_OUT / "checkpoint_best_ema.pth"
assert finetuned_ckpt.exists(), f"expected checkpoint at {finetuned_ckpt}"

FINAL_CKPT = CHECKPOINTS / "object_detection" / "rfdetr_nano_odvirat.pth"
import shutil as _sh
_sh.copy(finetuned_ckpt, FINAL_CKPT)
print("saved checkpoint ->", FINAL_CKPT, f"({FINAL_CKPT.stat().st_size / 1e6:.1f} MB)")

model_ft = RFDETRNano(pretrain_weights=str(FINAL_CKPT), num_classes=4)
print("reloaded fine-tuned model, class_names:", model_ft.class_names)


## 2.2 Validation: zero-shot COCO weights vs fine-tuned checkpoint

Same OD-VIRAT val images, same IoU-matching function from Part 1, two models. The fine-tuned
model outputs OD-VIRAT's own 4 class names directly (no COCO-name mapping needed), including
**Carrying_object** -- a class the zero-shot COCO model cannot detect at all (no COCO equivalent),
so that row is `N/A` for zero-shot and a real number for fine-tuned.

In [ ]:
# === EXTENDED SECTION ===

def evaluate_detection_v2(model_obj, class_names, n_images=80, iou_thresh=0.5, conf_thresh=0.4,
                           class_map=None):
    """class_map: optional dict {model_class_id: odvirat_class_name} for the zero-shot COCO
    model. If None, model_obj.predict's class_id is assumed to already index class_names
    (true for the fine-tuned model)."""
    ann = json.loads((ODVIRAT / "json_anntations" / "val_annotations.json").read_text())
    cat_id_to_name = {c["id"]: c["name"] for c in ann["categories"]}
    img_by_id = {im["id"]: im for im in ann["images"]}
    gt_by_image = {}
    for a in ann["annotations"]:
        gt_by_image.setdefault(a["image_id"], []).append((cat_id_to_name[a["category_id"]], a["bbox"]))

    image_ids = list(gt_by_image)[:n_images]
    per_class = {c: dict(tp=0, fp=0, fn=0) for c in class_names}
    t0 = time.time()
    for img_id in image_ids:
        info = img_by_id[img_id]
        frame = cv2.imread(str(ODVIRAT / "data" / "val" / info["file_name"]))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        from PIL import Image
        detections = model_obj.predict(Image.fromarray(rgb), threshold=conf_thresh)

        gts = gt_by_image[img_id][:]
        matched = set()
        preds = []
        for box, cid, conf in zip(detections.xyxy, detections.class_id, detections.confidence):
            name = class_map.get(int(cid)) if class_map is not None else class_names[int(cid)]
            if name is None:
                continue
            x1, y1, x2, y2 = box
            preds.append((name, [x1, y1, x2 - x1, y2 - y1]))

        for name, box in preds:
            best_iou, best_j = 0.0, -1
            for j, (gname, gbox) in enumerate(gts):
                if j in matched or gname != name:
                    continue
                i = iou(box, gbox)
                if i > best_iou:
                    best_iou, best_j = i, j
            if best_iou >= iou_thresh:
                matched.add(best_j); per_class[name]["tp"] += 1
            elif name in per_class:
                per_class[name]["fp"] += 1
        for j, (gname, _) in enumerate(gts):
            if j not in matched and gname in per_class:
                per_class[gname]["fn"] += 1
    elapsed = time.time() - t0

    rows = []
    for c, d in per_class.items():
        tp, fp, fn = d["tp"], d["fp"], d["fn"]
        has_support = (tp + fn) > 0 or (class_map is None)
        if not has_support and class_map is not None:
            rows.append(dict(cls=c, precision=None, recall=None, f1=None, tp=tp, fp=fp, fn=fn))
            continue
        p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)
        rows.append(dict(cls=c, precision=p, recall=r, f1=f1, tp=tp, fp=fp, fn=fn))
    df = pd.DataFrame(rows)
    df.attrs["wall_time_s"] = elapsed
    df.attrs["fps"] = len(image_ids) / elapsed
    return df

ODVIRAT_CLASSES = ["Bike/Bicycle", "Car", "Carrying_object", "Person"]
zero_shot_class_map = {}
for cid, name in COCO_ID_TO_NAME.items():
    for ov_class, coco_set in ODVIRAT_TO_COCO.items():
        if name in coco_set:
            zero_shot_class_map[cid] = ov_class

zero_shot_df = evaluate_detection_v2(model, ODVIRAT_CLASSES, n_images=80, class_map=zero_shot_class_map)
zero_shot_df["variant"] = "zero-shot (COCO weights)"
finetuned_df = evaluate_detection_v2(model_ft, ODVIRAT_CLASSES, n_images=80, class_map=None)
finetuned_df["variant"] = "fine-tuned (this checkpoint)"

comparison_df = pd.concat([zero_shot_df, finetuned_df], ignore_index=True)
comparison_df


In [ ]:
# === EXTENDED SECTION ===

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
metrics = ["precision", "recall", "f1"]
x = range(len(ODVIRAT_CLASSES))
width = 0.35
for ax, metric in zip(axes, metrics):
    zs = [zero_shot_df[zero_shot_df.cls == c][metric].values[0] for c in ODVIRAT_CLASSES]
    ft = [finetuned_df[finetuned_df.cls == c][metric].values[0] for c in ODVIRAT_CLASSES]
    zs = [0 if v is None else v for v in zs]
    ax.bar([i - width/2 for i in x], zs, width, label="zero-shot", color="#a0a0a0")
    ax.bar([i + width/2 for i in x], ft, width, label="fine-tuned", color="#2563eb")
    ax.set_xticks(list(x)); ax.set_xticklabels(ODVIRAT_CLASSES, rotation=30, ha="right")
    ax.set_title(metric); ax.set_ylim(0, 1)
axes[0].legend()
fig.suptitle("Object detection: zero-shot COCO weights vs fine-tuned checkpoint, per OD-VIRAT class")
fig.tight_layout()
fig.savefig(FIGDIR / "object_detection_before_after.png", dpi=120)
plt.show()
print("Note: 'Carrying_object' zero-shot bars are 0 by construction -- COCO has no equivalent class,")
print("so the zero-shot model can never predict it. This is the concrete gain fine-tuning buys.")


## 2.3 Precision-recall curve (fine-tuned model, real val data)

Sweeps the confidence threshold and recomputes precision/recall at each point -- the standard
PR-curve read of `plan.md`'s M7 metric, not a single-threshold snapshot.

In [ ]:
# === EXTENDED SECTION ===

pr_points = []
for thresh in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    df = evaluate_detection_v2(model_ft, ODVIRAT_CLASSES, n_images=50, conf_thresh=thresh, class_map=None)
    tp = df.tp.sum(); fp = df.fp.sum(); fn = df.fn.sum()
    p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
    pr_points.append(dict(threshold=thresh, precision=p, recall=r))
pr_df = pd.DataFrame(pr_points)
pr_df


In [ ]:
# === EXTENDED SECTION ===

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(pr_df.recall, pr_df.precision, "o-", color="#2563eb")
for _, row in pr_df.iterrows():
    ax.annotate(f"{row.threshold:.1f}", (row.recall, row.precision),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("PR curve, fine-tuned RF-DETR Nano on OD-VIRAT val\n(labels = confidence threshold)")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(FIGDIR / "object_detection_pr_curve.png", dpi=120)
plt.show()


## 2.4 Qualitative validation on real, unlabeled video

OD-VIRAT above is real surveillance video, but as independent labeled stills, not a continuous
clip. This section runs the fine-tuned checkpoint on an actual real DVR video
(`DVR_NVR_IP Camera01...mp4`) frame by frame. **No ground truth exists for this clip**, so there
is no precision/recall here -- the graphs are detection count over time and confidence
distribution, which is what "does this hold up on real video" looks like without invented
labels.

In [ ]:
# === EXTENDED SECTION ===

def run_on_real_video(video_path, model_obj, class_names, sample_every_s=1.0, conf_thresh=0.4):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = max(1, int(round(fps * sample_every_s)))
    rows = []
    i = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if i % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            from PIL import Image
            detections = model_obj.predict(Image.fromarray(rgb), threshold=conf_thresh)
            for cid, conf in zip(detections.class_id, detections.confidence):
                if 0 <= int(cid) < len(class_names):
                    rows.append(dict(t_s=i / fps, cls=class_names[int(cid)], conf=float(conf)))
        i += 1
    cap.release()
    return pd.DataFrame(rows)

real_video_detections = run_on_real_video(HIKVISION_CLIP, model_ft, ODVIRAT_CLASSES)
print(f"{len(real_video_detections)} detections across the real clip")
real_video_detections.head(10)


In [ ]:
# === EXTENDED SECTION ===

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts_by_t = real_video_detections.groupby("t_s").size()
axes[0].plot(counts_by_t.index, counts_by_t.values, drawstyle="steps-post", color="#2563eb")
axes[0].set_xlabel("time (s) into clip"); axes[0].set_ylabel("detections in frame")
axes[0].set_title("Detection count over time (real, unlabeled video)")

axes[1].hist(real_video_detections["conf"], bins=15, color="#2563eb", edgecolor="white")
axes[1].set_xlabel("confidence"); axes[1].set_ylabel("count")
axes[1].set_title("Confidence distribution, real-video detections")
fig.tight_layout()
fig.savefig(FIGDIR / "object_detection_real_video_validation.png", dpi=120)
plt.show()


## 2.5 Motion analysis: tune and save a day/night profile

MOG2 has no learned weights -- there is nothing to "fine-tune" in the object-detection sense.
The honest equivalent of a checkpoint here is the **tuned `variance_threshold`**: this cell
sweeps it separately on a daylight and a night sequence (both real CDnet video, both with real
pixel-wise ground truth) and saves whichever value wins each category. This is what closes the
gap Part 1 found (naive MOG2 collapsing to 10% precision at night from IR noise).

In [ ]:
# === EXTENDED SECTION ===

def evaluate_motion_with_threshold(seq_dir, variance_threshold, subsample=8):
    start, end = map(int, (seq_dir / "temporalROI.txt").read_text().split())
    roi_mask = cv2.imread(str(seq_dir / "ROI.bmp"), cv2.IMREAD_GRAYSCALE) > 0
    n_frames = len(list((seq_dir / "input").glob("*.jpg")))
    bgsub = cv2.createBackgroundSubtractorMOG2(varThreshold=variance_threshold, detectShadows=True)
    tp = fp = fn = 0
    for i in range(1, n_frames + 1):
        frame = cv2.imread(str(seq_dir / "input" / f"in{i:06d}.jpg"))
        fg = bgsub.apply(frame)
        if i < start or i > end or i % subsample != 0:
            continue
        gt = cv2.imread(str(seq_dir / "groundtruth" / f"gt{i:06d}.png"), cv2.IMREAD_GRAYSCALE)
        valid = roi_mask & (gt != 85) & (gt != 170)
        pred_fg = (fg == 255) & valid
        gt_fg = (gt == 255) & valid
        tp += np.logical_and(pred_fg, gt_fg).sum()
        fp += np.logical_and(pred_fg, ~gt_fg).sum()
        fn += np.logical_and(~pred_fg, gt_fg).sum()
    p = tp / (tp + fp + 1e-9); r = tp / (tp + fn + 1e-9)
    return 2 * p * r / (p + r + 1e-9)

THRESHOLDS = [16, 30, 50, 80, 120, 160]
day_seq = CDNET / "baseline" / "pedestrians"
night_seq = CDNET / "nightVideos" / "streetCornerAtNight"

day_sweep = [dict(variance_threshold=vt, f1=evaluate_motion_with_threshold(day_seq, vt)) for vt in THRESHOLDS]
night_sweep = [dict(variance_threshold=vt, f1=evaluate_motion_with_threshold(night_seq, vt)) for vt in THRESHOLDS]
day_sweep_df = pd.DataFrame(day_sweep)
night_sweep_df = pd.DataFrame(night_sweep)

best_day = day_sweep_df.loc[day_sweep_df.f1.idxmax()]
best_night = night_sweep_df.loc[night_sweep_df.f1.idxmax()]
print("best day variance_threshold:", best_day.variance_threshold, "F1:", round(best_day.f1, 3))
print("best night variance_threshold:", best_night.variance_threshold, "F1:", round(best_night.f1, 3))


In [ ]:
# === EXTENDED SECTION ===

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(day_sweep_df.variance_threshold, day_sweep_df.f1, "o-", label="day (pedestrians)", color="#2563eb")
ax.plot(night_sweep_df.variance_threshold, night_sweep_df.f1, "o-", label="night (streetCornerAtNight)", color="#dc2626")
ax.axvline(16, color="gray", linestyle="--", label="OpenCV default (16)")
ax.set_xlabel("MOG2 variance_threshold"); ax.set_ylabel("F1 vs ground truth")
ax.set_title("Motion F1 vs threshold -- day/night need different values")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "motion_threshold_tuning.png", dpi=120)
plt.show()


In [ ]:
# === EXTENDED SECTION ===

motion_profile = {
    "day": {"variance_threshold": float(best_day.variance_threshold), "f1_on_validation": float(best_day.f1)},
    "night": {"variance_threshold": float(best_night.variance_threshold), "f1_on_validation": float(best_night.f1)},
    "default_opencv": {"variance_threshold": 16.0},
    "note": "Tuned on CDnet2014 baseline/pedestrians (day) and nightVideos/streetCornerAtNight (night). "
            "Consumed by dvr_scan.subtractor.SubtractorMOG2(variance_threshold=...). "
            "Day/night selection at runtime should use frame brightness or a channel schedule, "
            "not this notebook -- this file only records which value to use once that's decided.",
}
profile_path = CHECKPOINTS / "motion_analysis" / "motion_profile.json"
profile_path.write_text(json.dumps(motion_profile, indent=2))
print("saved ->", profile_path)
print(json.dumps(motion_profile, indent=2))


## 2.6 Cross-camera correlation: offset recovery, visualized

Same synthetic-pairing setup as Part 1, with a histogram of recovered offsets across the
nearest-match pairs, so the spread (not just the median) is visible.

In [ ]:
# === EXTENDED SECTION ===

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(nearest["delta_s"], bins=20, color="#2563eb", edgecolor="white")
ax.axvline(CAMERA_B_OFFSET_S, color="#dc2626", linestyle="--", label=f"injected offset ({CAMERA_B_OFFSET_S}s)")
ax.axvline(nearest["delta_s"].median(), color="#16a34a", linestyle="-", label=f"recovered median ({nearest['delta_s'].median():.2f}s)")
ax.set_xlabel("recovered delta (s)"); ax.set_ylabel("count of nearest-match pairs")
ax.set_title("Cross-camera correlation: recovered offset distribution")
ax.legend()
fig.tight_layout()
fig.savefig(FIGDIR / "cross_camera_offset_recovery.png", dpi=120)
plt.show()


## 2.7 Criminal / face detection -- checkpoint, with an explicit data-gap caveat

**This is not the same kind of section as the two above, and that's stated plainly rather than
papered over.** `criminal_detection/face_utils.py` (already in the Niriksha codebase) builds a
gallery by clustering face encodings from a folder of images into unique individuals and pickling
the result (`cluster_and_encode_faces`) -- that pipeline is exercised below, for real, on faces
that already appear in this corpus's own footage.

What this section is **not**: a "criminal" identification system. No public dataset of identified
criminal suspects exists, and building one from random bystanders in stock CCTV footage would be
mislabeling real, identifiable people -- not something to fabricate for a demo. The people whose
faces get clustered below are anonymous bystanders in the Hikvision office clip, labeled only
`person_A`/`person_B`-style cluster IDs, never a name. Per `plan.md` line 379: *"Faces are leads,
not identification."* A real deployment populates this gallery from case-specific photos supplied
by investigators under proper authorization -- this notebook cannot and does not do that part.

In [ ]:
# === EXTENDED SECTION ===

try:
    import face_recognition
    FACE_RECOGNITION_AVAILABLE = True
except ImportError as e:
    FACE_RECOGNITION_AVAILABLE = False
    print("face_recognition not importable in this environment:", e)
    print("(dlib has no prebuilt wheel for this Python build here and needs a C++ toolchain to")
    print(" compile from source -- this is an environment limitation, not a code issue. The")
    print(" pipeline below is real and already in criminal_detection/face_utils.py; it just")
    print(" cannot run to produce real output on this machine right now.)")


In [ ]:
# === EXTENDED SECTION ===

if FACE_RECOGNITION_AVAILABLE:
    import sys
    sys.path.insert(0, str(REPO_ROOT / "Niriksha" / "Niriksha-main" / "dvr-scan-py"))
    from criminal_detection.face_utils import cluster_and_encode_faces

    # Extract a handful of real face crops from the office clip's own frames (anonymous bystanders).
    face_frames_dir = CHECKPOINTS / "criminal_detection" / "_face_crops"
    face_frames_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(HIKVISION_CLIP))
    fps = cap.get(cv2.CAP_PROP_FPS)
    saved = 0
    i = 0
    while saved < 20:
        ret, frame = cap.read()
        if not ret:
            break
        if i % int(fps * 2) == 0:
            cv2.imwrite(str(face_frames_dir / f"frame_{saved:03d}.jpg"), frame)
            saved += 1
        i += 1
    cap.release()
    print(f"saved {saved} full frames for face extraction (whole frames -- cluster_and_encode_faces")
    print("locates faces itself via face_recognition.face_encodings)")

    gallery_path = CHECKPOINTS / "criminal_detection" / "face_gallery.pkl"
    cluster_and_encode_faces(str(face_frames_dir), str(gallery_path))
    print("saved ->", gallery_path)
else:
    gallery_path = None


## 2.8 Checkpoint manifest

One file the frontend (or anyone else) can read to find every artifact this notebook produced,
without re-deriving anything.

In [ ]:
# === EXTENDED SECTION ===

manifest = {
    "generated_by": "notebooks/m7_analytics_demo.ipynb",
    "object_detection": {
        "checkpoint": str(FINAL_CKPT.relative_to(REPO_ROOT)),
        "base_model": "RFDETRNano",
        "classes": ODVIRAT_CLASSES,
        "trained_on": f"{n_train} images, OD-VIRAT Tiny train split (subset)",
        "epochs": 3,
        "validation": {
            "dataset": "OD-VIRAT Tiny val split (labeled, real surveillance stills)",
            "per_class_f1": {row.cls: (None if row.f1 is None else round(float(row.f1), 3))
                              for row in finetuned_df.itertuples()},
        },
        "figures": [
            "notebooks/figures/object_detection_before_after.png",
            "notebooks/figures/object_detection_pr_curve.png",
            "notebooks/figures/object_detection_real_video_validation.png",
        ],
    },
    "motion_analysis": {
        "checkpoint": str(profile_path.relative_to(REPO_ROOT)),
        "method": "OpenCV MOG2 (not a learned model -- tuned threshold only)",
        "validation": "CDnet2014 baseline/pedestrians (day), nightVideos/streetCornerAtNight (night)",
        "figures": ["notebooks/figures/motion_threshold_tuning.png"],
    },
    "criminal_detection": {
        "checkpoint": (str(gallery_path.relative_to(REPO_ROOT)) if gallery_path and gallery_path.exists() else None),
        "status": "built" if (gallery_path and gallery_path.exists()) else "NOT built -- face_recognition/dlib unavailable in this environment",
        "caveat": "Gallery entries are anonymous bystander clusters (person_A/person_B-style IDs), "
                  "never names or real identities. A real 'criminal' watchlist requires "
                  "case-specific, investigator-authorized photos -- out of scope for any public "
                  "dataset or automated notebook.",
    },
    "cross_camera_correlation": {
        "figures": ["notebooks/figures/cross_camera_offset_recovery.png"],
        "note": "synthetic camera-B pairing (see Part 1, section 3) -- validates the matching logic, not real two-camera accuracy",
    },
}
manifest_path = CHECKPOINTS / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print("saved ->", manifest_path)
print(json.dumps(manifest, indent=2))
